In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [2]:
# data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_1990m11_BID.dta") # para bases de stata
data = pd.read_stata(r"Z:\survey\ECU\ENEMDU\2000\m11\data_orig\ECU_2000m11.dta") # para bases de stata

In [3]:
df, meta = pd.read_stata(r"datos/ECU_2000m11_BID.dta", iterator=True), None
meta = df.variable_labels()
print("\nVariable labels:")
for col, label in meta.items():
    print(f"{col}: {label}")


Variable labels:
region_BID_c: Regiones BID
region_c: 
pais_c: Nombre del PaÃ­s
anio_c: Anio de la encuesta
mes_c: Mes de la encuesta
zona_c: Zona del pais
factor_ch: Factor de expansion del hogar
idh_ch: ID del hogar
idp_ci: ID de la persona en el hogar
factor_ci: Factor de expansion del individuo
sexo_ci: Sexo del individuo
edad_ci: Edad del individuo en aÃ±os
relacion_ci: Relacion o parentesco con el jefe del hogar
civil_ci: Estado civil
jefe_ci: Jefe/a de hogar
nconyuges_ch: # de conyuges en el hogar
nhijos_ch: # de hijos en el hogar
notropari_ch: # de otros familiares en el hogar
notronopari_ch: # de no familiares en el hogar
nempdom_ch: # de empleados domesticos
clasehog_ch: Tipo de hogar
nmiembros_ch: # de miembros en el hogar
miembros_ci: =1: es miembro del hogar
nmayor21_ch: # de familiares mayores a 21 anios en el hogar
nmenor21_ch: # de familiares menores a 21 anios en el hogar
nmayor65_ch: # de familiares mayores a 65 anios en el hogar
nmenor6_ch: # de familiares menores a

## Revisar los datos

- rgnal - regional
- area - area
- rn - región natural
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- ingpat - Ingresos como patrono o cuenta propia
- ingasa - ingreso líquido por salario
- ingasa1 - cuál fue su ingreso total por sueldo
- asa - recibio por trabajo especies, alimentos etc.
- ingasa2 - monto recibido por trabajo especies
- ingsec - ocup. secundaria cual fué su ingreso por sueldo
- inginv - monto de ingresos derivados de bienes de capital
- ingjub - Ingresos por jubilación o pensión
- ingotr - por otros ingresos
- ingbon - ingreso por bono solidario
- fexp - factor de expansión
- ingrl - ingresos

En esta encuesta desaparecen 'ingasg' e 'ingepv' que eran el ingreso laboral monetario, desaparece también 'ingdom', y aparecen las variables 'ingasa', 'ingasa1', 'asa', 'ingasa2' e 'ingsec' qeu son ingresos del trabajo asalariado de la actividad primaria y secundaria monetario y no monetario, se manteine ingrl como ingreso total, para mantener la misma línea que con los años anteriores vamos solo a usar el ingreso del trabajo monetario de la actividad principal

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba trabajando tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

Hay un problema con las variables 'ingasa' e 'ingasa1', los promedios son demasiado diferentes, puede ser que 'ingasa1' tenga valores en sucres aún cuando la entrevista ya se hizo en dólares, así que usaremos 'ingasa', dejamos por fuera el ingreso de salarios de la actividad secundaria por motivos de mantener comparable con los años anteriores 

In [3]:
data[['ingasa', 'ingasa1', 'ingsec']].mean()

ingasa     847627.145795
ingasa1    552770.885090
ingsec     310668.258659
dtype: float64

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62469 entries, 0 to 62468
Columns: 104 entries, rgnal to b1004
dtypes: category(55), float32(3), float64(30), int16(1), int8(5), object(10)
memory usage: 23.5+ MB


Filtramos solo las columnas de interés para alivar el peso en la memoria

In [5]:
data.columns

Index(['rgnal', 'area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar',
       'formul', 'persona',
       ...
       'vivefe', 'ingrl', 'secins', 'subequ', 'fexp', 'condact', 'peamsiu',
       'idhogar', 'id_persona', 'b1004'],
      dtype='object', length=104)

In [6]:
data = data[['rgnal', 'area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda',
             'hogar', 'persona', 'numpers', 'edad', 'ingpat', 'ingasa',
             'ingasa1', 'asa', 'ingasa2', 'ingsec', 'inginv', 'ingjub',
             'ingotr', 'ingbon', 'fexp', 'ingrl', 'ene', 'feb', 'mar', 
             'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']]

Creamos una variable de ingreso laboral quitando de la lista los valores m'as grandes, esta encuesta ya se hizo en dólares y no tienen sentido algunos valores, así que los limpiamos

       4150 |          1        0.00       99.13
       5000 |          1        0.00       99.13
       9999 |          3        0.01       99.14
      99999 |          3        0.01       99.14
    9999999 |          5        0.01       99.15
   9.00e+07 |          2        0.00       99.16
   1.00e+08 |        470        0.84      100.00

In [15]:
data['ingasa'] = data['ingasa'].replace(9999, np.nan)
data['ingasa'] = data['ingasa'].replace(99999, np.nan)
data['ingasa'] = data['ingasa'].replace(9999999, np.nan)
data['ingasa'] = data['ingasa'].replace(99999999, np.nan)
data['ingasa'] = data['ingasa'].replace(9.00e+07, np.nan)
data['ingasa'] = data['ingasa'].replace(1.00e+08, np.nan)

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados en un mes, ahora las variables categoricas por mes tienen diferentes leyendas
- desocupado y sin buscar trabajo
- trabajando
- buscando trabajo
- 0.0

In [9]:
data['dic'].value_counts()

dic
desocupado y sin buscar trabajo    29646
trabajando                         24743
buscando trabajo                    1267
0.0                                   66
Name: count, dtype: int64

In [10]:
data['ingr_ene'] = data.apply(lambda x: x['ingr'] if x['ene'] == 'trabajando' else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingr'] if x['feb'] == 'trabajando' else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingr'] if x['mar'] == 'trabajando' else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingr'] if x['abr'] == 'trabajando' else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingr'] if x['may'] == 'trabajando' else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingr'] if x['jun'] == 'trabajando' else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingr'] if x['jul'] == 'trabajando' else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingr'] if x['ago'] == 'trabajando' else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingr'] if x['sep'] == 'trabajando' else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingr'] if x['oct'] == 'trabajando' else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingr'] if x['nov'] == 'trabajando' else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingr'] if x['dic'] == 'trabajando' else None, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [11]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2000]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [12]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [13]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

Diccionario ciudades disponibles

In [14]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

In [15]:
data['ciudad_asignada'].value_counts()

ciudad_asignada
Nacional     35420
Guayaquil    14130
Quito         7728
Cuenca        5191
Name: count, dtype: int64

### Asignamos el ipc correspondiente según trimestre y ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [16]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [17]:
data['ipc_t1'] = data.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data['ipc_base_t1'] = data.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data['ipc_t2'] = data.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data['ipc_base_t2'] = data.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data['ipc_t3'] = data.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data['ipc_base_t3'] = data.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data['ipc_t4'] = data.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data['ipc_base_t4'] = data.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [18]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

In [19]:
# Ingreso real por mes
data['ingr_ene_r'] = data['ingr_ene'] * data['def_t1']
data['ingr_feb_r'] = data['ingr_feb'] * data['def_t1']
data['ingr_mar_r'] = data['ingr_mar'] * data['def_t1']
data['ingr_abr_r'] = data['ingr_abr'] * data['def_t2']
data['ingr_may_r'] = data['ingr_may'] * data['def_t2']
data['ingr_jun_r'] = data['ingr_jun'] * data['def_t2']
data['ingr_jul_r'] = data['ingr_jul'] * data['def_t3']
data['ingr_ago_r'] = data['ingr_ago'] * data['def_t3']
data['ingr_sep_r'] = data['ingr_sep'] * data['def_t3']
data['ingr_oct_r'] = data['ingr_oct'] * data['def_t4']
data['ingr_nov_r'] = data['ingr_nov'] * data['def_t4']
data['ingr_dic_r'] = data['ingr_dic'] * data['def_t4']

Ingreso mensual promedio en el trimeste

In [20]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

## Calculo ingreso de los hogares

In [21]:
columnas_idef = ['rgnal', 'area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda',
             'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

13963

In [22]:
data[['rgnal', 'area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda',
             'hogar', 'idef_hogar', 'persona', 'numpers']]

,rgnal,area,rn,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,4,1,1,010150,001,004,01,1,411010150001004011,4,4
1,4,1,1,010150,001,004,01,1,411010150001004011,3,4
2,4,1,1,010150,001,004,01,1,411010150001004011,1,4
3,4,1,1,010150,001,004,01,1,411010150001004011,2,4
4,4,1,1,010150,001,004,09,1,411010150001004091,2,6
...,...,...,...,...,...,...,...,...,...,...,...
62464,3,2,1,050160,999,015,07,1,321050160999015071,1,5
62465,3,2,1,050160,999,015,07,1,321050160999015071,4,5
62466,3,2,1,050160,999,015,07,1,321050160999015071,5,5
62467,3,2,1,050160,999,015,07,1,321050160999015071,3,5


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [23]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [24]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [25]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']].mean()

ingr_t1_h     223.31767
ingr_t2_h    226.704075
ingr_t3_h    228.508807
ingr_t4_h    228.852181
dtype: object

In [26]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  228.85218132663147
Mediana del ingreso de un hogar t4:  124.10870869708424


## Sacamos edades negativas y mayores a 100 años

In [27]:
len(data)

62469

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [28]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [29]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

62469

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [30]:
k = 0.4
s = 0.9

In [31]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [32]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [33]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [34]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']].mean()

ingr_t_t1     60.22773
ingr_t_t2    61.101356
ingr_t_t3    61.501438
ingr_t_t4    61.593861
dtype: object

In [35]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  61.59386086063949
Mediana del ingreso individual descontando cargas familiares t4:  31.312886975032946


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [36]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [37]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [38]:
data['persona_fexp'] = 1 * data['fexp']

In [39]:
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    data[col_pobres] = (
        (data[col_ingr] - umbral_dict.get(t)) < 0
    ).astype(int)

In [40]:
print("pobreza t1: ", (data['pobres_t1'] * data['fexp']).sum()/data.loc[data['ingr_t_t1'] >= 0]['persona_fexp'].sum())
print("pobreza t2: ", (data['pobres_t2'] * data['fexp']).sum()/data.loc[data['ingr_t_t2'] >= 0]['persona_fexp'].sum())
print("pobreza t3: ", (data['pobres_t3'] * data['fexp']).sum()/data.loc[data['ingr_t_t3'] >= 0]['persona_fexp'].sum())
print("pobreza t4: ", (data['pobres_t4'] * data['fexp']).sum()/data.loc[data['ingr_t_t4'] >= 0]['persona_fexp'].sum())

pobreza t1:  0.8324120051603283
pobreza t2:  0.831279331101491
pobreza t3:  0.8308372082240649
pobreza t4:  0.833055799239789


In [41]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [42]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.832412,0.603881,0.513696,NaN,NaN,NaN,NaN
t2,0.831279,0.600609,0.50993,NaN,NaN,NaN,NaN
t3,0.830837,0.600518,0.509258,NaN,NaN,NaN,NaN
t4,0.833056,0.603748,0.512995,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [43]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [44]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.832412,0.603881,0.513696,0.240812,0.493787,0.806817,NaN
t2,0.831279,0.600609,0.50993,0.239028,0.490318,0.802924,NaN
t3,0.830837,0.600518,0.509258,0.238459,0.488657,0.800435,NaN
t4,0.833056,0.603748,0.512995,0.240054,0.491971,0.804324,NaN


Guardamos el ingreso promedio

In [45]:
for t in [1, 2, 3, 4]:
    df_temp = data.loc[data[f'ingr_t_t{t}'] >= 0].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada

In [46]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.832412,0.603881,0.513696,0.240812,0.493787,0.806817,66.227346
t2,0.831279,0.600609,0.50993,0.239028,0.490318,0.802924,67.303726
t3,0.830837,0.600518,0.509258,0.238459,0.488657,0.800435,67.839175
t4,0.833056,0.603748,0.512995,0.240054,0.491971,0.804324,67.855487


In [47]:
datos_final.to_csv('datos_final.csv')